In [1]:
import pandas as pd
import numpy as np
import zipfile

zip_path = "df_r.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("df_r.csv") as f:
        df = pd.read_csv(f)

print(df.shape)

(37755, 20)


In [2]:
import re

def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()

    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)

    count = 0
    for person in people:
        # print(person)
        if '105' in person:
            count += 1
    # print("end")
    return count

In [3]:
df['killer_count'] = df['names'].apply(count_killers)
df['killer_count'].value_counts()

killer_count
1    37755
Name: count, dtype: int64

In [4]:
import re

def estimate_victims(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "па" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0


text1 = "Петров И.И. - ст.105 ч.1 УК РФ"
text2 = "Иванов С.С. - ст.105 ч.2 п.а, п.ж УК РФ"
text3 = "Сидоров - ст. 105 ч.2 п.п.а УК РФ"

print(estimate_victims(text1)) 
print(estimate_victims(text2))
print(estimate_victims(text3))

0
1
1


In [5]:
df['many_victims'] = df['names'].apply(estimate_victims)
df['many_victims'].value_counts()

many_victims
0    35687
1     2068
Name: count, dtype: int64

In [6]:
df["entryDate"] = pd.to_datetime(df["entryDate"])
df["year"] = df["entryDate"].dt.year

def get_season(month):
    if month in [12, 1, 2]:
        return "Зима"
    elif month in [3, 4, 5]:
        return "Весна"
    elif month in [6, 7, 8]:
        return "Лето"
    else:
        return "Осень"

df["season"] = df["entryDate"].dt.month.apply(get_season)

In [7]:
df.columns

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'killer_count', 'gender_accused',
       'prior_convictions', 'alcohol', 'precrime_argument', 'has_woman_victim',
       'has_man_victim', 'method', 'motive', 'location', 'prison_term',
       'many_victims', 'year', 'season'],
      dtype='object')

In [9]:
df = df.drop(['link_text', 'accused', 'articles', 'entryDate'], axis=1)

In [10]:
df['decision'].value_counts()

decision
Вынесен ПРИГОВОР                                        37601
Вступило в силу                                           152
Применены ПРИНУДИТЕЛЬНЫЕ МЕРЫ МЕДИЦИНСКОГО ХАРАКТЕРА        2
Name: count, dtype: int64

In [11]:
df.columns

Index(['id', 'region', 'names', 'judge', 'decision', 'killer_count',
       'gender_accused', 'prior_convictions', 'alcohol', 'precrime_argument',
       'has_woman_victim', 'has_man_victim', 'method', 'motive', 'location',
       'prison_term', 'many_victims', 'year', 'season'],
      dtype='object')

In [12]:
import re

def estimate_attempt(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)
        if "30" in cleaned:
            flag = True
    if flag:
        return 1
    else:
        return 0

text1 = "Петров И.И. - ст.30 ч.1 УК РФ"
text2 = "Иванов С.С. - ст.105 ч.2 п.а, п.ж УК РФ"
text3 = "Сидоров - ст. 105 ч.2 п.п.а УК РФ"

print(estimate_attempt(text1)) 
print(estimate_attempt(text2))
print(estimate_attempt(text3))

1
0
0


In [13]:
df['attempt'] = df['names'].apply(estimate_attempt)
df['attempt'].value_counts()

attempt
0    29228
1     8527
Name: count, dtype: int64

In [14]:
import re

def estimate_victim_child_or_helpless(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "в" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0

df['victim_child_or_helpless'] = df['names'].apply(estimate_victim_child_or_helpless)
df['victim_child_or_helpless'].value_counts()

victim_child_or_helpless
0    37131
1      624
Name: count, dtype: int64

In [15]:
import re

def estimate_cruelty(names_text: str) -> int:
    if not isinstance(names_text, str):
        return 0

    pattern = r"ст\. ?\d+(?: ч\.\d+)?(?: п\.[п.]?[\w.,]+)?"
    articles = re.findall(pattern, names_text)

    flag = False

    for article in articles:
        cleaned = re.sub(r"[.\s]", "", article).lower()
        #print(cleaned)

        if "105" in cleaned:
            if "ч2" in cleaned:
                if "д" in cleaned:
                    flag = True
    if flag:
        return 1
    else:
        return 0

df['cruel'] = df['names'].apply(estimate_cruelty)
df['cruel'].value_counts()

cruel
0    37056
1      699
Name: count, dtype: int64

In [16]:
df['region'].value_counts()

region
50    1805
2     1524
66    1522
24    1333
3     1242
      ... 
95      40
87      38
83      22
20       1
91       1
Name: count, Length: 86, dtype: int64

In [17]:
df['region'].min(), df['region'].max()

(1, 95)

In [18]:
region_map = {
    1: "Республика Адыгея",
    4: "Республика Алтай",
    2: "Республика Башкортостан",
    3: "Республика Бурятия",
    5: "Республика Дагестан",

    6: "Республика Ингушетия",
    7: "Кабардино-Балкарская Республика",
    8: "Республика Калмыкия",
    9: "Карачаево-Черкесская Республика",
    10: "Республика Карелия",

    11: "Республика Коми",
    82: "Республика Крым",
    12: "Республика Марий Эл",
    13: "Республика Мордовия",
    14: "Республика Саха (Якутия)",

    15: "Республика Северная Осетия — Алания",
    16: "Республика Татарстан",
    17: "Республика Тыва",
    18: "Удмуртская Республика",
    19: "Республика Хакасия",

    20: "Чеченская Республика",
    95: "Чеченская Республика",
    21: "Чувашская Республика",
    22: "Алтайский край",
    75: "Забайкальский край",
    41: "Камчатский край",

    23: "Краснодарский край",
    24: "Красноярский край",
    59: "Пермский край",
    25: "Приморский край",
    26: "Ставропольский край",

    27: "Хабаровский край",
    28: "Амурская область",
    29: "Архангельская область",
    30: "Астраханская область",
    31: "Белгородская область",

    32: "Брянская область",
    33: "Владимирская область",
    34: "Волгоградская область",
    35: "Вологодская область",
    36: "Воронежская область",

    37: "Ивановская область",
    38: "Иркутская область",
    39: "Калининградская область",
    40: "Калужская область",
    42: "Кемеровская область",

    43: "Кировская область",
    44: "Костромская область",
    45: "Курганская область",
    46: "Курская область",
    47: "Ленинградская область",

    48: "Липецкая область",
    49: "Магаданская область",
    50: "Московская область",
    51: "Мурманская область",
    52: "Нижегородская область",

    53: "Новгородская область",
    54: "Новосибирская область",
    55: "Омская область",
    56: "Оренбургская область",
    57: "Орловская область",

    58: "Пензенская область",
    60: "Псковская область",
    61: "Ростовская область",
    62: "Рязанская область",
    63: "Самарская область",

    64: "Саратовская область",
    65: "Сахалинская область",
    66: "Свердловская область",
    67: "Смоленская область",
    68: "Тамбовская область",

    69: "Тверская область",
    70: "Томская область",
    71: "Тульская область",
    72: "Тюменская область",
    73: "Ульяновская область",

    74: "Челябинская область",
    76: "Ярославская область",
    77: "Москва",
    78: "Санкт-Петербург",
    92: "Севастополь",

    79: "Еврейская автономная область",
    83: "Ненецкий автономный округ",
    80: "Донецкая народная республика",
    81: "Луганская народная республика",
    84: "Херсонская область",

    85: "Запорожская область",
    86: "Ханты-Мансийский автономный округ — Югра",
    87: "Чукотский автономный округ",
    89: "Ямало-Ненецкий автономный округ",
    94: "Байконур"
}

df["region_name"] = df["region"].map(region_map)
df["region_name"].value_counts()

region_name
Московская область                     1805
Республика Башкортостан                1524
Свердловская область                   1522
Красноярский край                      1333
Республика Бурятия                     1242
                                       ... 
Республика Северная Осетия — Алания      67
Республика Адыгея                        62
Чеченская Республика                     41
Чукотский автономный округ               38
Ненецкий автономный округ                22
Name: count, Length: 84, dtype: int64

In [19]:
for val, count in df["region_name"].value_counts().items():
    print(f"{val}, {count}")

Московская область, 1805
Республика Башкортостан, 1524
Свердловская область, 1522
Красноярский край, 1333
Республика Бурятия, 1242
Кемеровская область, 1178
Краснодарский край, 1154
Иркутская область, 1088
Пермский край, 1029
Нижегородская область, 1004
Челябинская область, 1000
Забайкальский край, 997
Алтайский край, 838
Ростовская область, 821
Хабаровский край, 794
Новосибирская область, 770
Архангельская область, 751
Приморский край, 659
Оренбургская область, 574
Санкт-Петербург, 562
Республика Саха (Якутия), 542
Республика Татарстан, 541
Ленинградская область, 540
Ставропольский край, 528
Саратовская область, 523
Омская область, 514
Тульская область, 471
Владимирская область, 442
Республика Коми, 430
Удмуртская Республика, 424
Волгоградская область, 424
Курганская область, 414
Тюменская область, 406
Вологодская область, 385
Республика Крым, 384
Ульяновская область, 378
Воронежская область, 356
Республика Тыва, 356
Кировская область, 345
Ивановская область, 324
Чувашская Республика,

In [20]:
df.columns

Index(['id', 'region', 'names', 'judge', 'decision', 'killer_count',
       'gender_accused', 'prior_convictions', 'alcohol', 'precrime_argument',
       'has_woman_victim', 'has_man_victim', 'method', 'motive', 'location',
       'prison_term', 'many_victims', 'year', 'season', 'attempt',
       'victim_child_or_helpless', 'cruel', 'region_name'],
      dtype='object')

In [21]:
df = df.drop(['names', 'killer_count', 'region'], axis=1)

In [22]:
df.columns

Index(['id', 'judge', 'decision', 'gender_accused', 'prior_convictions',
       'alcohol', 'precrime_argument', 'has_woman_victim', 'has_man_victim',
       'method', 'motive', 'location', 'prison_term', 'many_victims', 'year',
       'season', 'attempt', 'victim_child_or_helpless', 'cruel',
       'region_name'],
      dtype='object')

In [23]:
df.to_csv('df_labeled_all_final22.csv', index=False)

In [24]:
df.shape

(37755, 20)